# CS5246 Text Mining Final Project

## Introduction
This project focuses on analyzing articles and posts from the Singapore subreddit to gain insights into public opinions, trending topics, and sentiment patterns. To ensure reliable and meaningful analysis, the raw dataset undergoes a structured preprocessing workflow, transforming noisy social media text into clean, standardized, and linguistically annotated data.

### Project Structure

```bash
CS5246Project/
    │
    ├─ data/
    │   ├─ inverted_index/
    │   │   ├─ inverted_index_tfidf_titles.json
    │   │   └─ inverted_index_tfidf_fulltext.json
    │   │
    │   ├─ models/
    │   │   ├─ sentence_bert_model/
    │   │   ├─ bm25_comments_model.joblib
    │   │   ├─ bm25_fulltext_model.joblib
    │   │   ├─ bm25_titles_model.joblib
    │   │   ├─ tfidf_comments_vectorizer.joblib
    │   │   └─ tfidf_posts_vectorizer.joblib
    │   │
    │   ├─ vector_database/
    │   │   ├─ bert_comments_embeddings.npy
    │   │   ├─ bert_contents_embeddings.npy
    │   │   ├─ bert_fulltext_embeddings.npy
    │   │   ├─ bert_titles_embeddings.npy
    │   │   ├─ bm25_comments.npz
    │   │   ├─ bm25_contents.npz
    │   │   ├─ bm25_fulltext.npz
    │   │   ├─ bm25_titles.npz
    │   │   ├─ tfidf_comments.npz
    │   │   ├─ tfidf_contents.npz
    │   │   ├─ tfidf_fulltext.npz
    │   │   └─ tfidf_titles.npz
    │   │
    │   ├─ PostVault.csv
    │   ├─ CommentVault.csv
    │   └─ raw_data/
    │
    ├─ sentiment_plots/
    │   ├─ emotion_plots/
    │   ├─ emotion_dashboard.py
    │   └─ plot_emotion_summary.py
    │
-----------------------------------------------------------------
│   ├─ Stage_0_Introduction.ipynb                               │
-----------------------------------------------------------------            
    ├─ Stage_1_Data_Collection_and_Data_Cleaning.ipynb
    ├─ Stage_2_POS_and_NER_Tagging.ipynb
    ├─ Stage_3_Singlish_Normalisation.ipynb
    ├─ Stage_4_Singlish_to_English_Conversion.ipynb     
    ├─ Stage_5_Common_Normalisation.ipynb
    ├─ Stage_6_Vector_Space_Model_and_Inverted_Index.ipynb
    ├─ Stage_7_Sentiment_Analysis.ipynb
    ├─ Stage_8_Clustering_and_Visualization.ipynb       
    ├─ Stage_9_Document_Search.ipynb
    └─ utilities/
        │
        ├─ pp_class.py
        ├─ singlish_dictionary.json
        ├─ singlish_regex_to_text.txt
        └─ slang_dictionary.csv
```

### Workflow
```
[Stage 1: Data Collection and Data Cleaning]
  - Input:
    * Raw dataset

  - Operations:
    1. Remove URLs
    2. Stripping user mentions
    3. Reducing character elongation
    4. Normalizing punctuation
    5. Remove bots and mods
    6. De-deduplicate
    7. Remove deleted posts
    8. Remove 3 word posts
    9. Expand contractions
    10. Convert emoji to text

  - Output:
    * PostVault.csv
    * CommentVault.csv

  - Notes:
    * Posts:
      1. Stored separately as `title` and `content`
      2. Combined version stored as `fulltext`
      3. Cleaned outputs stored in `cleaned_title` and `cleaned_content`
    * Comments:
      1. Original text stored as `text`
      2. Cleaned version stored as `cleaned_text`

        │
        │
        │
        ▼

[Stage 2: POS and NER Tagging]
  - Operations:
    1. Create POS and NER Tagger

        │    
        │
        │
        ▼

[Stage 3: Singlish Normalisation]
  - Input:
    * PostVault.csv
    * CommentVault.csv

  - Operations:
    1. Normalize Singlish expressions into standardized Singlish forms to reduce lexical variation and ensure consistency

  - Note:
    * Posts:
      1. Output stored in `singlish_normalized_title` and `singlish_normalized_content`
    * Comments:
      1. Output stored in `singlish_normalized_text`

        │
        │
        │
        ▼

[Stage 4: Singlish to English Conversion]
  - Input:
    * PostVault.csv
    * CommentVault.csv

  - Operations:
    1. Convert Singlish expressions into standard English to support advanced NLP processing  
    2. Track the number of converted Singlish terms and store in the `singlish_count` column (for downstream ranking or analysis)  

  - Note:
    * Posts:
      - Output stored in `english_converted_title` and `english_converted_content`
    * Comments:
      - Output stored in `english_converted_text`

        │
        │
        │
        ▼

[Stage 5: Text / Short Message Normalisation]
  - Input:
    * PostVault.csv
    * CommentVault.csv

  - Operations:
    1. Expand Shortforms and Slang: Expand short forms and slang expressions into their full forms
    2. Handling Emojis: Handle emojis by converting them into meaningful textual representations or removing them if irrelevant
    3. Spelling Correction: Perform spelling correction to standardize words
    4. Stop Word Removal: Remove stop words to reduce noise in the text
    5. Lemmatization: Apply lemmatization to reduce words to their base or root form
    6. Final Cleaning and Title–Content Concatenation: Perform final text cleaning and concatenate the title and content fields into a unified text representation for downstream processing

  - Output:
    * Posts:
      - Output stored in `expanded_title`, `demojized_title`, `spellchecked_title`, `lemmatized_title`, `expanded_content`, `demojized_content`, `spellchecked_content`, `lemmatized_content`, and `lemmatized_full_text`
    * Comments:
      - Output stored in `expanded_text`, `demojized_text`, `spellchecked_text`, `lemmatized_text`

        │
        │
        │
        ▼

[Stage 6: Vector Space Model and Inverted Index]
  - Input:
    * PostVault.csv
    * CommentVault.csv
    - Note: The processed text columns from Stage 5 (e.g., lemmatized_title, lemmatized_content, lemmatized_full_text, cleaned_text)

  - Operations:
    1. TF-IDF Vectorization:
      a. Creates a TfidfVectorizer for posts and fits it to lemmatized_full_text.
      b. Transforms lemmatized_title, lemmatized_content, and lemmatized_full_text into TF-IDF matrices.
      c. Creates and fits a separate TfidfVectorizer for comments to cleaned_text.
    2. Save TF-IDF Matrices: Saves the generated TF-IDF sparse matrices (tfidf_titles_matrix, tfidf_contents_matrix, tfidf_full_text_matrix, tfidf_comments_matrix) to .npz files.
    3. Save TF-IDF Vectorizers: Saves the fitted TfidfVectorizer models (tfidf_posts_vectorizer, tfidf_comments_vectorizer) to .joblib files.
    4. BM25 Tokenization: Tokenizes the lemmatized_title, lemmatized_content, lemmatized_full_text, and cleaned_text fields for BM25 processing.
    5. BM25 Model Initialization: Initializes BM25Okapi models for tokenized titles, full text, and comments.
    6. Save BM25 Models: Saves the initialized BM25Okapi models (bm25_titles, bm25_fulltext, bm25_comments) to .joblib files.
    7. Generate BM25 Matrices: Builds sparse BM25 score matrices (bm25_titles_matrix, bm25_contents_matrix, bm25_full_text_matrix, bm25_comments_matrix) from the tokenized documents and BM25 models.
    8. Save BM25 Matrices: Saves the generated BM25 sparse matrices to .npz files.
    9. Load Pre-trained BERT Model: Loads a SentenceTransformer model (all-MiniLM-L6-v2).
    10. Generate BERT Embeddings: Generates dense BERT embeddings for title, content, and fulltext fields of posts.
    11. Save BERT Embeddings: Saves the generated BERT embeddings (bert_titles_embeddings, bert_contents_embeddings, bert_full_text_embeddings) to .npy files.
    12. Save BERT Model: Saves the loaded SentenceTransformer model to a directory.
    13. Build Inverted Index Function: Defines a function to build an inverted index from a TF-IDF matrix and its feature names.
    14. Generate Inverted Indices: Uses the defined function to create inverted indices for TF-IDF processed titles (inverted_index_titles) and full text (inverted_index_full_text).
    15. Save Inverted Indices: Saves the generated inverted indices to .json files.

  - Output:
    * Posts:
      - tfidf_titles.npz: TF-IDF matrix for post titles.
      - tfidf_contents.npz: TF-IDF matrix for post content.
      - tfidf_fulltext.npz: TF-IDF matrix for post full text.
      - bm25_titles.npz: BM25 score matrix for post titles.
      - bm25_contents.npz: BM25 score matrix for post content.
      - bm25_fulltext.npz: BM25 score matrix for post full text.
      - bert_titles_embeddings.npy: BERT embeddings for post titles.
      - bert_contents_embeddings.npy: BERT embeddings for post content.
      - bert_fulltext_embeddings.npy: BERT embeddings for post full text.
      - inverted_index_tfidf_titles.json: Inverted index for TF-IDF post titles.
      - inverted_index_tfidf_fulltext.json: Inverted index for TF-IDF post full text.
      - tfidf_posts_vectorizer.joblib: Trained TF-IDF vectorizer for posts.
      - bm25_titles_model.joblib: Trained BM25 model for post titles.
      - bm25_fulltext_model.joblib: Trained BM25 model for post full text.
    * Comments:
      - tfidf_comments.npz: TF-IDF matrix for comments.
      - bm25_comments.npz: BM25 score matrix for comments.
      - tfidf_comments_vectorizer.joblib: Trained TF-IDF vectorizer for comments.
      - bm25_comments_model.joblib: Trained BM25 model for comments.
    * General Models:
      - sentence_bert_model/: Directory containing the saved Sentence-BERT model.

        │
        │
        │
        ▼

[Stage 7: Sentiment Analysis]
  - Input:
    * PostVault.csv
    * CommentVault.csv

  - Operations:

  - Output:

        │
        │
        │
        ▼

[Stage 8: Clustering and Visualization]
  - Input:
    * PostVault.csv
    * CommentVault.csv
    * bert_comments_embeddings.npy
    * bert_contents_embeddings.npy
    * bert_fulltext_embeddings.npy
    * bert_titles_embeddings.npy
    * bm25_comments.npz
    * bm25_contents.npz
    * bm25_fulltext.npz
    * bm25_titles.npz
    * tfidf_comments.npz
    * tfidf_contents.npz
    * tfidf_fulltext.npz
    * tfidf_titles.npz
    * tfidf_comments_vectorizer.joblib
    * tfidf_posts_vectorizer.joblib

  - Operations:
    1. Load TF-IDF, BM25, and BERT feature matrices  
    2. Apply Truncated SVD to reduce the dimensionality of the high-dimensional feature space  
    3. Use the Silhouette Score to determine the optimal number of clusters (K)  
    4. Perform K-means clustering on the reduced feature set  
    5. Store the resulting cluster labels in `PostVault.csv` and `CommentVault.csv`  
    6. Perform stratified sampling based on cluster distribution  
    7. Apply t-SNE on the sampled data for visualization  
    8. Conduct word cloud analysis to identify prominent terms within clusters  
    9. Perform keyword extraction to identify representative terms for each cluster

  - Output:
    * Posts:
      - Output stored in `tfidf_cluster`, `bm25_cluster`, and `bert_cluster`
    * Comments:
      - Output stored in `tfidf_cluster`, `bm25_cluster`, and `bert_cluster`

        │
        │
        │
        ▼

[Stage 9: Document Search]
  - Input:
    * PostVault.csv
    * CommentVault.csv
    * bert_comments_embeddings.npy
    * bert_contents_embeddings.npy
    * bert_fulltext_embeddings.npy
    * bert_titles_embeddings.npy
    * bm25_comments.npz
    * bm25_contents.npz
    * bm25_fulltext.npz
    * bm25_titles.npz
    * tfidf_comments.npz
    * tfidf_contents.npz
    * tfidf_fulltext.npz
    * tfidf_titles.npz
    * inverted_index_tfidf_titles.json
    * inverted_index_tfidf_fulltext.json
    * sentence_bert_model/
    * bm25_comments_model.joblib
    * bm25_fulltext_model.joblib
    * bm25_titles_model.joblib
    * tfidf_comments_vectorizer.joblib
    * tfidf_posts_vectorizer.joblib

  - Operations:
    1. Build a search engine using scores stored in the inverted index and cosine similarity within a vector space model (TF-IDF, BM25, and BERT embeddings). Incorporate additional business and heuristic rules, such as assigning higher weights to titles over content, filtering out posts with removed content, and prioritizing posts based on recency, number of comments, and upvote ratio  
    2. Build a recommendation system using scores stored in the inverted index and cosine similarity within a vector space model (TF-IDF, BM25, and BERT embeddings). Incorporate additional business and heuristic rules, such as filtering out posts with removed content and prioritizing posts based on recency, number of comments, and upvote ratio
```

### To access the mounted folder directly.

#### Step 1: Add the Shared Folder to your Google Drive (if you haven't already)

1.  **Open Google Drive** (drive.google.com) in your web browser.
2.  Go to the **'Shared with me'** section.
3.  Locate the folder that was shared with you.
4.  **Right-click** on the shared folder.
5.  Select **'Add shortcut to Drive'** (or 'Add to My Drive', depending on the interface). Choose a location within 'My Drive' if prompted, or simply add it to the root of 'My Drive'.

This creates a shortcut to the shared folder in your 'My Drive', making it accessible when Colab mounts your personal Drive.

#### Step 2: Access the Shared Folder in Colab (No action needed)

Once your Drive is mounted, you can navigate to the shared folder. If you added a shortcut, it will appear in your 'My Drive' just like any other folder. The path will typically be `/content/gdrive/MyDrive/YourSharedFolderName`.

It should be able to run the script below already

In [ ]:
# -----------
# Mount Drive
# -----------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Import Essential Libraries

In [ ]:
import os
import pandas as pd

### Path

In [ ]:
models_dir = '/content/drive/MyDrive/CS5246Project/data/models'
output_base_dir = '/content/drive/MyDrive/CS5246Project/data/intermediate_data'
inverted_index_dir = '/content/drive/MyDrive/CS5246Project/data/inverted_index'
vector_database_dir = '/content/drive/MyDrive/CS5246Project/data/vector_database'

### Utilities

In [ ]:
def display_first_text_column_head(dfs_input, description):
    """
    Displays the head of the 'cleaned_text' column from the first DataFrame in a list
    (or a single DataFrame) that contains this column and has non-empty values.
    This function is designed to quickly inspect the content of processed text data.

    Args:
        dfs_input (list or pandas.DataFrame): A single DataFrame or a list of DataFrames
                                              to search for the 'cleaned_text' column.
        description (str): A string used in the print statement to describe the DataFrames
                           being inspected (e.g., 'raw posts', 'cleaned comments').

    Returns:
        None: This function only prints output to the console and does not return any value.

    The function iterates through the provided DataFrames, checks for the presence
    and non-emptiness of a 'cleaned_text' column, and displays its head if found.
    If no such column is found or if all found columns are empty, it prints a
    corresponding message.
    """
    # Normalize input to always be a list of DataFrames
    if not isinstance(dfs_input, list):
        dfs = [dfs_input]
    else:
        dfs = dfs_input

    found_text = False
    for df in dfs:
        if 'cleaned_text' in df.columns and not df['cleaned_text'].empty:
            print(f"\nFirst few rows of {description} 'cleaned_text' column from a relevant DataFrame:")
            display(df['cleaned_text'].head())
            found_text = True
            break

    if not found_text:
        print(f"No 'cleaned_text' columns with content found in any of the {description} DataFrames to display.")

def save_dfs_to_csv(dfs_list, output_dir, prefix):
    """
    Saves a list of pandas DataFrames to CSV files within a specified output directory.
    Each DataFrame is saved with a filename indicating whether it contains 'posts' or 'comments'
    based on the presence of a 'title' column.

    Args:
        dfs_list (list of pandas.DataFrame): A list of DataFrames to be saved.
        output_dir (str): The path to the directory where the CSV files will be saved. If the directory does not exist, it will be created.
        prefix (str): A string prefix to be added to the filename of each saved CSV file.

    Returns:
        None: This function performs file I/O and does not return any value.

    Each DataFrame in `dfs_list` is processed. The function checks for a 'title' column
    to determine if the DataFrame represents 'posts' or 'comments'. The files are named
    in the format `prefix_df_type_index.csv` (e.g., `initial_posts_0.csv`).
    """

    os.makedirs(output_dir, exist_ok=True)
    print(f"Saving DataFrames to '{output_dir}' with prefix '{prefix}'...")
    for i, df in enumerate(dfs_list):

        df_type = "posts" if 'title' in df.columns else "comments"
        filename = f"{prefix}_{df_type}_{i}.csv"
        filepath = os.path.join(output_dir, filename)
        df.to_csv(filepath, index=False)
        print(f"Saved {df_type} DataFrame {i} to {filepath}")

def save_single_df_to_csv(df, output_dir, name):
    """
    Saves a pandas DataFrames to CSV file within a specified output directory.
    The DataFrame is saved with a filename indicating whether it contains 'posts' or 'comments'
    based on the presence of a 'title' column.

    Args:
        dfs_list (list of pandas.DataFrame): A list of DataFrames to be saved.
        output_dir (str): The path to the directory where the CSV files will be saved. If the directory does not exist, it will be created.
        name (str): The CSV file name.

    Returns:
        None: This function performs file I/O and does not return any value.

    Each DataFrame in `dfs_list` is processed. The function checks for a 'title' column
    to determine if the DataFrame represents 'posts' or 'comments'. The files are named
    in the format `name.csv` (e.g., `PostVault.csv`).
    """

    os.makedirs(output_dir, exist_ok=True)
    print(f"Saving DataFrames to '{output_dir}' with name '{name}'...")
    # Determine if it's posts or comments based on column presence
    df_type = "posts" if 'title' in df.columns else "comments"
    filename = f"{name}.csv"
    filepath = os.path.join(output_dir, filename)
    df.to_csv(filepath, index=False)
    print("-" * 100)
    print(f"{name} saved to {filepath}")
    print("-" * 100)

In [ ]:
if __name__ == "__main__":
    print("---------")
    print("PostVault")
    print("---------")
    # Create an empty DataFrame for posts with a 'title' column

    empty_posts_df = pd.DataFrame(columns=[
        'index', 'id', 'title', 'content', 'score', 'upvote_ratio', 'num_comments',
        'created_utc', 'subreddit_id', 'has_emoji', 'year', 'month', 'day_of_week',
        'hour', 'score_bucket', 'text_length', 'word_count', 'singlish_count',
        'cleaned_text', 'singlish_normalized', 'english_converted', 'slang_expanded',
        'demojized', 'spelling_corrected', 'stopword_removed', 'lemmatized',
        'predicted_emotion', 'prob_anger', 'prob_disgust', 'prob_fear', 'prob_joy',
        'prob_neutral', 'prob_sadness', 'prob_surprise', 'tfidf_cluster', 'bm25_cluster'
        ])

    # Save the empty posts DataFrame as PostVault.csv
    save_single_df_to_csv(empty_posts_df, '/content/drive/MyDrive/CS5246Project/data', 'PostVault')

    print("------------")
    print("CommentVault")
    print("------------")
    # Create an empty DataFrame for comments (without a 'title' column)
    empty_comments_df = pd.DataFrame(columns=[
        'id'
        ])

    # Save the empty comments DataFrame as CommentVault.csv
    save_single_df_to_csv(empty_comments_df, '/content/drive/MyDrive/CS5246Project/data', 'CommentVault')

---------
PostVault
---------
Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data' with name 'PostVault'...
PostVault saved to /content/drive/MyDrive/CS5246Project/data/PostVault.csv
-----------
PostComment
-----------
Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data' with name 'CommentVault'...
CommentVault saved to /content/drive/MyDrive/CS5246Project/data/CommentVault.csv
